## Run GraphST

In [5]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse

In [6]:
adata = sc.read_visium('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data', count_file='filtered_feature_bc_matrix.h5')
ground_truth = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/metadata.tsv', delimiter='\t', index_col=0)
adata.obs['ground_truth'] = ground_truth['ground_truth']
adata.var_names_make_unique()

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [7]:
coords = pd.DataFrame(
    adata.obsm["spatial"],
    index=adata.obs_names,
    columns=["x", "y"]
)

In [8]:
X = adata.X

if sparse.issparse(X):
    X = X.tocsr()

# Save expression matrix
from scipy.io import mmwrite

mmwrite(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_counts.mtx",
    X
)

In [9]:
pd.DataFrame({
    "gene": adata.var_names
}).to_csv(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_genes.csv",
    index=False
)

In [10]:
obs = adata.obs.copy()

obs.to_csv(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_obs.csv"
)

In [11]:
coords.to_csv(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_coordinates.csv"
)

In [18]:
## RUN Bass with R script: data/hoan/spatial_transcriptomics/run_Bass_Breast_Cancer.R

In [19]:
# bass_labels = pd.read_csv('/data/hoan/spatial_transcriptomics/GraphBG-analysis/Data/Section_1_BASS_labels.csv')

In [16]:
# Read BASS results
bass_results = pd.read_csv(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_BASS_labels.csv"
)

# Use spot IDs as the index
bass_results = bass_results.set_index("spot_id")

# Check for missing or duplicated IDs
assert bass_results.index.is_unique, "Duplicate spot IDs in BASS results"

missing = adata.obs_names.difference(bass_results.index)
if len(missing) > 0:
    raise ValueError(
        f"{len(missing)} AnnData spots have no matching BASS label."
    )

# Align labels to AnnData spot order
adata.obs["BASS"] = bass_results["BASS_labels"].reindex(
    adata.obs_names
).astype("category")

# Verify
print(adata.obs["BASS"].value_counts())
print(adata.obs[["BASS"]].head())

# Save updated AnnData
adata.write_h5ad(
    "/data/hoan/spatial_transcriptomics/GraphBG-analysis/"
    "Data/Section_1_BASS.h5ad"
)

print("Saved AnnData with BASS labels.")

1     426
16    325
7     283
12    250
9     215
2     208
6     201
4     195
17    193
14    186
19    185
3     183
8     167
15    158
11    156
5     131
13    125
18    118
10     93
Name: BASS, dtype: int64
                   BASS
AAACAAGTATCTCCCA-1   17
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    4
AAACAGGGTCTATATT-1   11
AAACAGTGTTCCTGGG-1    1
Saved AnnData with BASS labels.
